# Benchmarks

## Initialize

In [1]:
import os
import math
import pathlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import clear_output

import warnings
from lifelines.utils import CensoringType
from lifelines.utils import concordance_index

In [2]:
node = !hostname
if "sc" in node[0]:
    base_path = "/sc-projects/sc-proj-ukb-cvd"
else: 
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"
print(base_path)

project_label = "22_medical_records"
project_path = f"{base_path}/results/projects/{project_label}"
figure_path = f"{project_path}/figures"
output_path = f"{project_path}/data"

experiment = 220413
experiment_path = f"{output_path}/{experiment}"
pathlib.Path(experiment_path).mkdir(parents=True, exist_ok=True)

/sc-projects/sc-proj-ukb-cvd


In [3]:
endpoints_md = pd.read_csv(f"{experiment_path}/endpoints.csv")
endpoints = sorted(endpoints_md.endpoint.to_list())

In [4]:
partitions = [p for p in range(0, 22)]
splits = ["train", "valid", "test"]

In [5]:
endpoint_defs = pd.read_feather(f"{output_path}/phecode_defs_220306.feather").query("endpoint==@endpoints").sort_values("endpoint").set_index("endpoint")

In [6]:
eligable_eids = pd.read_feather(f"{output_path}/eligable_eids_220306.feather")
eids_dict = eligable_eids.set_index("endpoint")["eid_list"].to_dict()

In [7]:
%env MKL_NUM_THREADS=4
%env NUMEXPR_NUM_THREADS=4
%env OMP_NUM_THREADS=4

env: MKL_NUM_THREADS=4
env: NUMEXPR_NUM_THREADS=4
env: OMP_NUM_THREADS=4


In [ ]:
ray.shutdown()

In [8]:
import ray

ray.init(num_cpus=24)#, dashboard_port=24762, dashboard_host="0.0.0.0", include_dashboard=True)#, webui_url="0.0.0.0"))

2022-04-21 13:56:35,086	INFO services.py:1412 -- View the Ray dashboard at http://127.0.0.1:8267


{'node_ip_address': '10.32.105.13',
 'raylet_ip_address': '10.32.105.13',
 'redis_address': None,
 'object_store_address': '/tmp/ray/session_2022-04-21_13-56-28_990900_444888/sockets/plasma_store',
 'raylet_socket_name': '/tmp/ray/session_2022-04-21_13-56-28_990900_444888/sockets/raylet',
 'webui_url': '127.0.0.1:8267',
 'session_dir': '/tmp/ray/session_2022-04-21_13-56-28_990900_444888',
 'metrics_export_port': 53591,
 'gcs_address': '10.32.105.13:64471',
 'address': '10.32.105.13:64471',
 'node_id': '531387f7a10b299b6451dcc1bbe21f94633520a1f9a57baa5fdf8e7a'}

# Train COX

In [9]:
in_path = pathlib.Path(f"{experiment_path}/coxph/input")
model_path = f"{experiment_path}/coxph/models"

out_path = f"{experiment_path}/coxph/predictions"
pathlib.Path(out_path).mkdir(parents=True, exist_ok=True)

In [10]:
import pickle
import zstandard

def load_pickle(fp):
    with open(fp, "rb") as fh:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(fh) as decompressor:
            data = pickle.loads(decompressor.read())
    return data

In [36]:
cox_paths = !ls $model_path
cox_paths = [p for p in cox_paths if "_MedicalHistory" in p or "+MedicalHistory" in p or "I(" in p]
cox = pd.Series(cox_paths).str.split("_", expand=True)\
    .assign(path = cox_paths)\
    .assign(endpoint = lambda x: x[0]+"_"+x[1])\
    .assign(score = lambda x: x[2])\
    .assign(partition = lambda x: x[3].str.replace(".p", "", regex=True).astype(int))\
    [["endpoint", "score", "partition", "path"]].sort_values(["endpoint", "score", "partition"])\
    .query("endpoint ==@ endpoints")\
    .reset_index(drop=True)
cox

,endpoint,score,partition,path
0,OMOP_4306655,Age+Sex+MedicalHistory,0,OMOP_4306655_Age+Sex+MedicalHistory_0.p
1,OMOP_4306655,Age+Sex+MedicalHistory,1,OMOP_4306655_Age+Sex+MedicalHistory_1.p
2,OMOP_4306655,Age+Sex+MedicalHistory,2,OMOP_4306655_Age+Sex+MedicalHistory_2.p
3,OMOP_4306655,Age+Sex+MedicalHistory,3,OMOP_4306655_Age+Sex+MedicalHistory_3.p
4,OMOP_4306655,Age+Sex+MedicalHistory,4,OMOP_4306655_Age+Sex+MedicalHistory_4.p
...,...,...,...,...
103945,phecode_979,MedicalHistory,17,phecode_979_MedicalHistory_17.p
103946,phecode_979,MedicalHistory,18,phecode_979_MedicalHistory_18.p
103947,phecode_979,MedicalHistory,19,phecode_979_MedicalHistory_19.p
103948,phecode_979,MedicalHistory,20,phecode_979_MedicalHistory_20.p


In [37]:
#endpoints = sorted(cox.endpoint.unique().tolist())
scores = sorted(cox.score.unique().tolist())
partitions = sorted(cox.partition.unique().tolist())

In [38]:
import ray

@ray.remote
def get_cox_info(p):
    cph = load_pickle(f"{model_path}/{p}")
    p_split = p.split("_")
    endpoint = f"{p_split[0]}_{p_split[1]}"
    score = p_split[2]
    partition = p_split[3][:-2]
    hrs = cph.hazard_ratios_.to_dict()
    
    if score=="Age+Sex+MedicalHistory+I(Age*MH)":
        hr_mh = hrs[endpoint.replace("-", "")]
        
        key_int_age = [k for k in hrs if "age_at_recruitment_f21022_0_0" in k and endpoint.replace("-", "") in k][0]
        hr_mh_age = hrs[key_int_age]
        
        try:
            key_int_sex = [k for k in hrs if "sex_f31_0_0" in k and endpoint.replace("-", "") in k][0]
            hr_mh_sex = hrs[key_int_sex]
        except:
            hr_mh_sex = np.nan
    else:
        hr_mh = hrs[endpoint] 
        hr_mh_age = np.nan
        hr_mh_sex = np.nan
        
    return {"endpoint": endpoint, "score": score, "partition": partition, "hrs": hrs, 
            "hrs_mh": hr_mh, 
            "hrs_mh_age": hr_mh_age, 
            "hrs_mh_sex": hr_mh_sex
           }

In [39]:
rows = []

for p in tqdm(cox.path.tolist()):
    rows.append(get_cox_info.remote(p))

  0%|          | 0/103950 [00:00<?, ?it/s]

In [40]:
rows = [ray.get(r) for r in tqdm(rows)]

  0%|          | 0/103950 [00:00<?, ?it/s]

In [33]:
rows[10]

{'endpoint': 'phecode_101-8',
 'score': 'Age+Sex+MedicalHistory',
 'partition': '10',
 'hrs': {'age_at_recruitment_f21022_0_0': 1.330651116430104,
  'sex_f31_0_0': 0.8560454376131976,
  'phecode_101-8': 4.22639172668347},
 'hrs_mh': 4.22639172668347,
 'hrs_mh_age': nan,
 'hrs_mh_sex': nan}

In [42]:
hrs_endpoints = pd.DataFrame({}).append(rows, ignore_index=True)

In [43]:
hrs_endpoints 

,endpoint,score,partition,hrs,hrs_mh,hrs_mh_age,hrs_mh_sex
0,OMOP_4306655,Age+Sex+MedicalHistory,0,{'age_at_recruitment_f21022_0_0': 1.6594235146...,4.543129,NaN,NaN
1,OMOP_4306655,Age+Sex+MedicalHistory,1,{'age_at_recruitment_f21022_0_0': 1.6388899956...,4.626850,NaN,NaN
2,OMOP_4306655,Age+Sex+MedicalHistory,2,{'age_at_recruitment_f21022_0_0': 1.6212401635...,4.587563,NaN,NaN
3,OMOP_4306655,Age+Sex+MedicalHistory,3,{'age_at_recruitment_f21022_0_0': 1.6194602523...,4.625650,NaN,NaN
4,OMOP_4306655,Age+Sex+MedicalHistory,4,{'age_at_recruitment_f21022_0_0': 1.5810022531...,4.494688,NaN,NaN
...,...,...,...,...,...,...,...
103945,phecode_979,MedicalHistory,17,{'phecode_979': 3.1042201319125984},3.104220,NaN,NaN
103946,phecode_979,MedicalHistory,18,{'phecode_979': 3.213847317655845},3.213847,NaN,NaN
103947,phecode_979,MedicalHistory,19,{'phecode_979': 2.9566621546108993},2.956662,NaN,NaN
103948,phecode_979,MedicalHistory,20,{'phecode_979': 3.1252990891129953},3.125299,NaN,NaN


In [44]:
name = f"hrs_endpoints"
hrs_endpoints.to_feather(f"{experiment_path}/{name}.feather")

In [ ]:
hrs_endpoints

In [ ]:
cph.plot()

In [ ]:
#[[]]

In [ ]:
cph.print_summary()

In [ ]:
models = ['Identity(AgeSex+Records)+MLP',
 'Identity(AgeSex)+MLP',
 'Identity(Records)+MLP']

In [ ]:
AgeSex = ["age_at_recruitment_f21022_0_0", "sex_f31_0_0"]

In [ ]:
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
import zstandard
import pickle

def get_features(endpoint):
    features = {
        'Identity(AgeSex)+MLP': {
            "AgeSex": [endpoint]},
        'Identity(Records)+MLP': {
            "MedicalHistory": [endpoint],
            "Age+Sex+MedicalHistory": AgeSex + [endpoint]},
        'Identity(AgeSex+Records)+MLP': {
            "AgeSexMedicalHistory": [endpoint]
        }
    }
    return features

def get_test_data(in_path, partition, models, mapping):
    train_data = {model: pd.read_feather(f"{in_path}/{model}/{partition}/test.feather").set_index("eid").replace(mapping)for model in models}
    return train_data
            
def load_pickle(fp):
    with open(fp, "rb") as fh:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(fh) as decompressor:
            data = pickle.loads(decompressor.read())
    return data

def predict_cox(cph, data_endpoint, endpoint, feature_set, partition, pred_path):
    times = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    time_cols = {t: f"Ft_{t}" for t in times}
    
    surv_test = 1-cph.predict_survival_function(data_endpoint, times=times) 
    temp_pred = data_endpoint.reset_index()[["eid"]].assign(endpoint=endpoint, features=feature_set, partition=partition)
    for t, col in time_cols.items(): temp_pred[col] = surv_test.T[t].to_list()
    
    temp_pred.to_feather(f"{out_path}/{endpoint}_{feature_set}_{partition}.feather")

@ray.remote
def predict_endpoint(data_partition, eids_dict, endpoint, partition, models, features, model_path, out_path):
    eids_incl = eids_dict[endpoint].tolist()
    results = []
    for model in models:
        data_model = data_partition[model]
        for feature_set, covariates in features[model].items():
            identifier = f"{endpoint}_{feature_set}_{partition}"
            pred_path = f"{out_path}/{identifier}.feather"
            cph = load_pickle(f"{model_path}/{identifier}.p")
            data_endpoint = data_model[data_model.index.isin(eids_incl)]
            predict_cox(cph, data_endpoint, endpoint, feature_set, partition, pred_path)
    return True

In [ ]:
mapping = {"sex_f31_0_0": {"Female":0, "Male":1}}

ray_eids = ray.put(eids_dict)
for partition in tqdm(partitions):
    try:
        del ray_partition
    except:
        print("Ray object not yet initialised")
    ray_partition = ray.put(get_test_data(in_path, partition, models, mapping))
    progress = []
    for endpoint in endpoints:
        features = get_features(endpoint)
        progress.append(predict_endpoint.remote(ray_partition, ray_eids, endpoint, partition, models, features, model_path, out_path))
    [ray.get(s) for s in tqdm(progress)]

In [ ]:
endpoint = endpoints[0]
partition=0

In [ ]:
eids_incl = eids_dict[endpoint].tolist()
features = get_features(endpoint)

In [ ]:
mapping = {"sex_f31_0_0": {"Female":0, "Male":1}}
data_partition = get_test_data(in_path, partition, models, mapping)

In [ ]:
for model in models:
    data_model = data_partition[model]
    for feature_set, covariates in features[model].items():
        identifier = f"{endpoint}_{feature_set}_{partition}"
        cph = load_pickle(f"{model_path}/{identifier}.p")
        data_endpoint = data_model[data_model.index.isin(eids_incl)]
        predict_cox(cph, data_endpoint, endpoint, feature_set, partition, f"{out_path}/{identifier}.feather")

In [ ]:
mapping = {"sex_f31_0_0": {"Female":0, "Male":1}}

ray_eids = ray.put(eids_dict)
ray_endpoint_defs = ray.put(endpoint_defs)
for partition in tqdm(partitions):
    try:
        del ray_partition
    except:
        print("Ray object not yet initialised")
    data_partition = get_test_data(in_path, partition, models, mapping)
    ray_partition = ray.put(data_partition)
    progress = []
    for endpoint in endpoints:
        progress.append(predict_endpoint.remote(ray_partition, ray_eids, endpoint, partition, models, model_path, out_path))
    [ray.get(s) for s in tqdm(progress)]

In [ ]:
def load_pickle(fp):
    with open(fp, "rb") as fh:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(fh) as decompressor:
            data = pickle.loads(decompressor.read())
    return data

In [ ]:
data_fit = load_pickle("/sc-projects/sc-proj-ukb-cvd/results/projects/22_medical_records/data/220309/coxph/errordata_phecode_054_Age+Sex+MedicalHistory_10.p").astype(np.float32)

In [ ]:
data_fit

In [ ]:
endpoint = "phecode_054"
cph = CoxPHFitter(penalizer=0)
cph.fit(data_fit, f"{endpoint}_time", f"{endpoint}_event", step_size=0.5, verbose=True)

In [ ]:
np.isnanf(test.values).sum(axis=0)

In [ ]:
test.dtypes

In [ ]:
test.isinf()

In [ ]:
test.isinf().sum(axis=0)

In [ ]:
[s for s in progress if s!=s]

In [ ]:
ray.get(progress[0])

In [ ]:
[ray.get(s) for s in tqdm(progress)]

In [ ]:
data_partition = get_train_data(in_path, 4, models, mapping)

In [ ]:
for model in models:
    print(model, data_partition[model].isna().sum().sum())
    t = data_partition[model].replace([np.inf, -np.inf], np.nan, inplace=False)
    print(len(data_partition[model])-len(t.dropna(how="all")))

In [ ]:
ray.get(progress[158])

In [ ]:
progress = !ls $out_path
len(progress)/len(endpoints)/4

In [ ]:
mapping = {"sex_f31_0_0": {"Female":0, "Male":1}}

for partition in tqdm(partitions[:1]):
    data_partition = get_partition_data(in_path, partition, models, mapping)
    ray_partition = ray.put(data_partition)
    for endpoint in tqdm(endpoints[:100]):
        fit_endpoint.remote(ray_partition, endpoint, partition, models)

In [ ]:
for endpoint in tqdm(endpoints[:10]):
    eids_incl = eids_dict[endpoint].tolist()
    features = get_features(endpoint)
    for model in models:
        data_model = data_partition[model]
        for feature_set, covariates in features[model].items():
            data_endpoint = data_model[covariates + [endpoint, f"{endpoint}_event", f"{endpoint}_time"]]
            data_endpoint = data_endpoint[data_endpoint.index.isin(eids_incl)]
            cph_path = f"{out_path}/{endpoint}_{feature_set}_{partition}.p"
            if not os.path.isfile(cph_path):
                cph = fit_cox(data_endpoint, endpoint, penalizer=0.0, step_size=0.1)
                pickle.dump(cph, open(cph_path, "wb"))

In [ ]:
cph = CoxPHFitter(penalizer=0)
cph.fit(data_endpoint, f"{endpoint}_time", f"{endpoint}_event", step_size=0.1, show_progress=True)

In [ ]:
data_endpoint

In [ ]:
for endpoint in tqdm(endpoints[:100]):
    eids_incl = eids_dict[endpoint].tolist()
    features = get_features(endpoint)
    for model in models:
        data_model = data_temp[model]
        for feature_set, covariates in features[model].items():
            data_endpoint = data_model[covariates + [endpoint, f"{endpoint}_event", f"{endpoint}_time"]]
            data_endpoint = data_endpoint[data_endpoint.index.isin(eids_incl)]

In [ ]:
data_endpoint

In [ ]:
eids_incl[0]

In [ ]:
def get_features(metabolomics, endpoint):
    features = {"COX": {"Metabolomics": metabolomics,
                        "Age+Sex": AgeSex, 
                        "ASCVDnoblood": ASCVDnoblood,
                        "ASCVD": ASCVD,
                        "PANELnoblood": PANELnoblood,
                        "PANELjustbloodcount": PANELjustbloodcount,
                        "PANEL": PANEL,
                       },
                "PCA": {"Metabolomics": [f'NMR_PCA{i}' for i in range(10)]},
                
                "DS":  {"Metabolomics": [f"logh_{endpoint}_Metabolomics"], 
                        "Age+Sex+Metabolomics": AgeSex+[f"logh_{endpoint}_Metabolomics"], 
                        "ASCVDnoblood+Metabolomics": ASCVDnoblood+[f"logh_{endpoint}_Metabolomics"],
                         "ASCVD+Metabolomics": ASCVD+[f"logh_{endpoint}_Metabolomics"],
                        "PANELnoblood+Metabolomics": PANELnoblood+[f"logh_{endpoint}_Metabolomics"],
                        "PANELjustbloodcount+Metabolomics": PANELjustbloodcount+[f"logh_{endpoint}_Metabolomics"],
                        "PANEL+Metabolomics": PANEL+[f"logh_{endpoint}_Metabolomics"],
                       }
               }
    
    return features

def train_cox(endpoints, partition, endpoint):
    features_all = get_all_features(metabolomics, endpoints)
    data_train = data_train[["eid", "partition", "split"]+features_all]
        for module in tqdm(modules):
            for endpoint in tqdm(endpoints):
                features = get_features(metabolomics, endpoint)
                eids_incl = eids_dict[endpoint] 
                train_endpoint = data_train.query("eid==@eids_incl")
                for partition in partitions:
                    train_partition = train_endpoint.query("partition==@partition")
                    frame_dict[f"{module}_{endpoint}_{partition}"] = fit_partition.remote(train_partition, endpoint, module, features, partition)
                    del train_partition
    return frame_dict

In [ ]:
from lifelines import CoxPHFitter
from dask.diagnostics import ProgressBar
from lifelines.exceptions import ConvergenceError
import pickle

def flatten_dict(features):
    df = pd.json_normalize(d, sep='_')
    t = list(df.to_dict(orient='records')[0].values())
    flat_list = [item for sublist in t for item in sublist]
    return list(set(flat_list))

def flatten_list(l):
    return [item for sublist in l for item in sublist]

def get_all_features(metabolomics, endpoints):
    f_dicts = flatten_list([[ds for ds in get_features(metabolomics, endpoint).values()] for endpoint in endpoints])
    fs = list(sorted(set(flatten_list(flatten_list([list(p.values()) for p in f_dicts])))))
    return fs + [f"{endpoint}_event" for endpoint in endpoints] + [f"{endpoint}_event_time" for endpoint in endpoints]

def get_features(metabolomics, endpoint):
    features = {"COX": {"Metabolomics": metabolomics,
                        "Age+Sex": AgeSex, 
                        "ASCVDnoblood": ASCVDnoblood,
                        "ASCVD": ASCVD,
                        "PANELnoblood": PANELnoblood,
                        "PANELjustbloodcount": PANELjustbloodcount,
                        "PANEL": PANEL,
                       },
                "PCA": {"Metabolomics": [f'NMR_PCA{i}' for i in range(10)]},
                
                "DS":  {"Metabolomics": [f"logh_{endpoint}_Metabolomics"], 
                        "Age+Sex+Metabolomics": AgeSex+[f"logh_{endpoint}_Metabolomics"], 
                        "ASCVDnoblood+Metabolomics": ASCVDnoblood+[f"logh_{endpoint}_Metabolomics"],
                         "ASCVD+Metabolomics": ASCVD+[f"logh_{endpoint}_Metabolomics"],
                        "PANELnoblood+Metabolomics": PANELnoblood+[f"logh_{endpoint}_Metabolomics"],
                        "PANELjustbloodcount+Metabolomics": PANELjustbloodcount+[f"logh_{endpoint}_Metabolomics"],
                        "PANEL+Metabolomics": PANEL+[f"logh_{endpoint}_Metabolomics"],
                       }
               }
    
    return features
    
def fit_cox(train, endpoint, penalizer, step_size):
    cph = CoxPHFitter(penalizer=penalizer)
    cph.fit(train, f"{endpoint}_event_time", f"{endpoint}_event", step_size=step_size, show_progress=True)
    return cph

@ray.remote
def fit_partition(train_partition, endpoint, module, features, partition):
    for feature_set, covariates in tqdm(features[module].items()):
        cph_path = f"{dump_path}/{module}_{endpoint}_{feature_set}_{partition}.p"
        if endpoint == "M_type_2_diabetes":
            if "diabetes1" in covariates: covariates = [c for c in covariates if c!="diabetes1"]
            if "diabetes2" in covariates: covariates = [c for c in covariates if c!="diabetes2"]
        if endpoint in ["M_prostate_cancer", "M_ovarian_cancer", "M_breast_cancer", "M_uterus_cancer"]:
            if "sex" in covariates: covariates = [c for c in covariates if c!="sex"]
        if not os.path.isfile(cph_path):
            try:
                cph = fit_cox(train_partition[covariates + [f"{endpoint}_event", f"{endpoint}_event_time"]], endpoint, penalizer=0.0, step_size=0.5)
                pickle.dump(cph, open(cph_path, "wb"))
            except ConvergenceError:
                print("ConvergenceError", module, endpoint, feature_set, partition, "problem: reduce step size")
                try:
                    cph = fit_cox(train_partition[covariates + [f"{endpoint}_event", f"{endpoint}_event_time"]], endpoint, penalizer=0.0, step_size=0.1)
                    pickle.dump(cph, open(cph_path, "wb"))
                    print("ConvergenceError", module, endpoint, feature_set, partition, "trying with reduced step size ... successfull")
                except:
                    print("ConvergenceError", module, endpoint, feature_set, partition, "trying with reduced step size ... failed")
    return True

def train_cox(endpoints, modules, partitions, data_train):
    frame_dict = {}
    features_all = get_all_features(metabolomics, endpoints)
    data_train = data_train[["eid", "partition", "split"]+features_all]
    with ProgressBar():   
        for module in tqdm(modules):
            for endpoint in tqdm(endpoints):
                features = get_features(metabolomics, endpoint)
                eids_incl = eids_dict[endpoint] 
                train_endpoint = data_train.query("eid==@eids_incl")
                for partition in partitions:
                    train_partition = train_endpoint.query("partition==@partition")
                    frame_dict[f"{module}_{endpoint}_{partition}"] = fit_partition.remote(train_partition, endpoint, module, features, partition)
                    del train_partition
    return frame_dict

In [ ]:
ray.shutdown()

In [ ]:
endpoints

In [ ]:
frame_dict = train_cox(endpoints, modules, partitions, data_train)

In [ ]:
ray.shutdown()

## Apply COX to data

In [ ]:
import joblib
@ray.remote
def get_cph(path): 
    with open(path,'rb') as f:
        cph = pickle.load(f)
    return cph#return pd.read_csv(f"{path[:-8]}.csv", index_col=0)

In [ ]:
ray.shutdown()

In [ ]:
import glob, os
cph_dict = {}
for path in tqdm(glob.glob(f"{dump_path}/*.p")):
    cph_dict[pathlib.Path(path).stem] = get_cph.remote(path)

In [ ]:
for path in tqdm(glob.glob(f"{dump_path}/*.p")):
    cph_dict[pathlib.Path(path).stem] = ray.get(cph_dict[pathlib.Path(path).stem])

In [ ]:
def predict_cox(endpoints, modules, partitions, data_test, cph_dict):
    times = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    time_cols = {t: f"Ft_{t}" for t in times}
    
    results = []
    test_preds = []
    for module in tqdm(modules):
        for endpoint in tqdm(endpoints):
            features = get_features(metabolomics, endpoint)
            eids_incl = eids_dict[endpoint] 
            test_endpoint = data_test.query("eid==@eids_incl")
            for partition in partitions:
                test_partition = test_endpoint.query("partition==@partition")
                for feature_set, covariates in features[module].items():
                    #print(endpoint, feature_set)
                    if endpoint == "M_type_2_diabetes":
                        if "diabetes1" in covariates: covariates = [c for c in covariates if c!="diabetes1"]
                        if "diabetes2" in covariates: covariates = [c for c in covariates if c!="diabetes2"]
                    if endpoint in ["M_prostate_cancer", "M_ovarian_cancer", "M_breast_cancer", "M_uterus_cancer"]:
                        if "sex" in covariates: covariates = [c for c in covariates if c!="sex"]
                    cph = cph_dict[f"{module}_{endpoint}_{feature_set}_{partition}"]
                    surv_test = 1-cph.predict_survival_function(test_partition[covariates+ [f"{endpoint}_event", f"{endpoint}_event_time"]], times=times) 
                    temp_pred = test_partition.reset_index()[["eid"]].assign(endpoint=endpoint, module=module, features=feature_set, partition=partition)
                    for t, col in time_cols.items(): temp_pred[col] = surv_test.T[t].to_list()
                    test_preds.append(temp_pred)
                    results.append({"endpoint": endpoint,
                                    "module": module,
                                    "features": feature_set,
                                    "HR_dict": cph.hazard_ratios_.to_dict(),
                                    "partition": partition,
                                    "cph": cph,
                                   })
    return results, test_preds

In [ ]:
results, test_preds = predict_cox(endpoints, modules, partitions, data_test, cph_dict)

## Save Predictions

In [ ]:
predictions = pd.concat(test_preds, axis=0).reset_index()

In [ ]:
predictions.to_feather(f"{data_results_path}/predictions_{run}_metabolomics.feather")

In [ ]:
predictions.value_counts(["module", "features"])

In [ ]:
predictions.value_counts(["endpoint"])

## Save COX models

In [ ]:
results_df = pd.DataFrame().append(results, ignore_index=True)

In [ ]:
results_df

In [ ]:
def get_hr(hr_dict, endpoint): 
    if f"logh_{endpoint}_Metabolomics" in hr_dict: 
        hr = hr_dict[f"logh_{endpoint}_Metabolomics"]
    else:
        hr = np.nan
    return hr

In [ ]:
import pickle
results_df["HR_metabolomics"] = [get_hr(hr_dict, endpoint) for endpoint, hr_dict in zip(results_df["endpoint"], results_df["HR_dict"])]
results_df.drop(columns=["cph"]).to_feather(f"{data_results_path}/cox_{run}_metabolomics.feather")

In [ ]:
#!ls -lah $data_results_path
!ls -lah $dump_path | tail

In [ ]:
# STORE HRs in different table


In [ ]:
import glob
import pickle
import re

In [ ]:
def _extract_logh(d):
    for k, v in d.items():
        if k.startswith('logh_'):
            return v

In [ ]:
res

In [ ]:
# load models and put HRs into a df:
df_dict = {"endpoint": [],
           "partition": [],
           "features": [],
           "HR_Metabolomics": []
          }

for fp in glob.glob(os.path.join(dump_path,"DS_*.p")):
    try:
        res = re.search("(M_[a-zA-Z_]+[2a-zA-Z_]+)_([3a-zA-Z+]+)_(\d+)", fp).groups()

        df_dict['endpoint'].append(res[0])
        df_dict['features'].append(res[1])
        df_dict['partition'].append(res[2])

        cph = pickle.load(open(fp, "rb"))
        HR = _extract_logh(cph.hazard_ratios_.to_dict())
    #     HR = cph.hazard_ratios_.to_dict()[f'logh_{res[0]}_Metabolomics']
        df_dict['HR_Metabolomics'].append(HR)
    except:
        print(fp)

In [ ]:
HR_df = pd.DataFrame.from_dict(df_dict, orient="columns")
HR_df['module'] = 'DS'

In [ ]:
HR_df.head()

In [ ]:
HR_df.to_feather(f"{data_results_path}/MET_HRs_210819_metabolomics.feather")

In [ ]:
results_df = results_df.assign(HR_logh=results_df['hazard_ratio'].apply(_extract_logh))

In [ ]:
results_df['features'].unique()

In [ ]:
results_df[["endpoint", "module", "features", 'HR_logh']].reset_index().to_feather(f"{data_results_path}/MET_HRs_{run}_metabolomics.feather")

In [ ]:
f"{data_results_path}/MET_HRs_{run}_metabolomics.feather"

In [ ]:
raise NotImplementedError()

In [ ]:
# Other stuff

In [ ]:
cph = results_df.set_index(["endpoint", "module", "features"]).loc[('M_MACE', "DS", "Age+Sex+Metabolomics"), "cph"].iloc[0]

In [ ]:
cph.print_summary()

In [ ]:
dir(cph)

In [ ]:
cph.summary["coef"]

In [ ]:
cph.baseline_cumulative_hazard_

In [ ]:
cph.baseline_hazard_

In [ ]:
dir(cph)

In [ ]:
cph.print_summary()

In [ ]:
import plotly.express as px
scores_plot = ["COX_AgeSex", "DeepSurv_AgeSexMetabolomics"]
temp = results_df.assign(score = lambda x: x.module + "_" + x.features).query("score==@scores_plot")
fig = px.scatter(temp, y="cindex", x="score", color="features", facet_col="endpoint", facet_col_wrap=12, template="plotly_dark",
               category_orders={"features": ["COX_Metabolomics", "DeepSurv_Metabolomics", 
                                             "COX_AgeSex", 
                                             "COX_AgeSexMetabolomics", "DeepSurv_AgeSexMetabolomics"]})
fig.update_xaxes(matches=None)
#fig.update_yaxes(matches=None)
fig.show("png", width=1500, height=1000)

In [ ]:
import plotly.express as px
temp = results_df.assign(score = lambda x: x.module + "_" + x.features)
fig = px.scatter(temp, y="cindex", x="score", color="features", facet_col="endpoint", facet_col_wrap=5, template="plotly_dark",
               category_orders={"features": ["COX_Metabolomics", "DeepSurv_Metabolomics", 
                                             "COX_AgeSex", 
                                             "COX_AgeSexMetabolomics", "DeepSurv_AgeSexMetabolomics"]})
fig.update_xaxes(matches=None)
#fig.update_yaxes(matches=None)
fig.show("png", width=1500, height=1000)

In [ ]:
import plotly.express as px
temp = results_df.assign(score = lambda x: x.module + "_" + x.features)
fig = px.scatter(temp, y="cindex", x="score", color="features", facet_col="endpoint", facet_col_wrap=5, template="plotly_dark",
               category_orders={"features": ["COX_Metabolomics", "DeepSurv_Metabolomics", 
                                             "COX_AgeSex", "DeepSurv_AgeSex",
                                             "COX_AgeSexMetabolomics", "DeepSurv_AgeSexMetabolomics"]})
fig.update_xaxes(matches=None)
#fig.update_yaxes(matches=None)
fig.show("png", width=1500, height=1000)

In [ ]:
dpath = "/data/analysis/ag-reils/steinfej/data/3_datasets_post/210730_imaging_visit"
for partition in range(5):
    for split in ["train", "valid", "test"]:
        ddf = pd.read_feather(f"{dpath}/partition_{partition}/{split}/data.feather")
        print(partition, split, len(ddf))

In [ ]:
data_X = pd.read_feather("/data/analysis/ag-reils/ag-reils-shared/cardioRS/data/3_datasets_post/210730_imaging_visit/data_merged.feather")

In [ ]:
data_X.set_index("uk_biobank_assessment_centre_2_0").index.value_counts().index.to_list()

In [ ]:
data_X.set_index("uk_biobank_assessment_centre_2_0").index.value_counts()

In [ ]:
import plotly.express as px
temp = results_df.assign(score = lambda x: x.module + "_" + x.features)
fig = px.scatter(temp, y="cindex", x="score", color="features", facet_col="endpoint", facet_col_wrap=5, template="plotly_dark",
               category_orders={"features": ["COX_Metabolomics", "DeepSurv_Metabolomics", 
                                             "COX_AgeSex", "DeepSurv_AgeSex",
                                             "COX_AgeSexMetabolomics", "DeepSurv_AgeSexMetabolomics"]})
fig.update_xaxes(matches=None)
#fig.update_yaxes(matches=None)
fig.show("png", width=1500, height=1000)

In [ ]:
import plotly.express as px
temp = results_df.assign(score = lambda x: x.module + "_" + x.features)
fig = px.scatter(temp, y="cindex", x="score", color="features", facet_col="endpoint", facet_col_wrap=5, template="plotly_dark",
               category_orders={"features": ["COX_Metabolomics", "DeepSurv_Metabolomics", 
                                             "COX_AgeSex", "DeepSurv_AgeSex",
                                             "COX_AgeSexMetabolomics", "DeepSurv_AgeSexMetabolomics"]})
fig.update_xaxes(matches=None)
#fig.update_yaxes(matches=None)
fig.show("png", width=1500, height=1000)

In [ ]:
import plotly.express as px
temp = results_df.assign(score = lambda x: x.module + "_" + x.features)
fig = px.scatter(temp, y="cindex", x="score", color="features", facet_col="endpoint", facet_col_wrap=5, template="plotly_dark",
               category_orders={"features": ["COX_Metabolomics", "DeepSurv_Metabolomics", 
                                             "COX_AgeSex", "DeepSurv_AgeSex",
                                             "COX_AgeSexMetabolomics", "DeepSurv_AgeSexMetabolomics"]})
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None)
fig.show("png", width=1500, height=1000)

In [ ]:
import plotly.express as px
temp = results_df.assign(score = lambda x: x.module + "_" + x.features)
fig = px.scatter(temp, y="cindex", x="score", color="features", facet_col="endpoint", facet_col_wrap=5, template="plotly_dark",
               category_orders={"features": ["COX_Metabolomics", "DeepSurv_Metabolomics", 
                                             "COX_AgeSex", "DeepSurv_AgeSex",
                                             "COX_AgeSexMetabolomics", "DeepSurv_AgeSexMetabolomics"]})
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None)
fig.show("png", width=1500, height=1000)

In [ ]:
import plotly.express as px
temp = results_df.assign(score = lambda x: x.module + "_" + x.features)
fig = px.scatter(temp, y="cindex", x="score", color="features", facet_col="endpoint", facet_col_wrap=5, template="plotly_dark",
               category_orders={"features": ["COX_Metabolomics", "DeepSurv_Metabolomics", 
                                             "COX_AgeSex", "DeepSurv_AgeSex",
                                             "COX_AgeSexMetabolomics", "DeepSurv_AgeSexMetabolomics"]})
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None)
fig.show("png", width=1500, height=1000)

In [ ]:
import plotly.express as px
temp = results_df.assign(score = lambda x: x.module + "_" + x.features)
fig = px.scatter(temp, y="cindex", x="score", color="features", facet_col="endpoint", facet_col_wrap=5, template="plotly_dark",
               category_orders={"features": ["COX_Metabolomics", "DeepSurv_Metabolomics", 
                                             "COX_AgeSex", "DeepSurv_AgeSex",
                                             "COX_AgeSexMetabolomics", "DeepSurv_AgeSexMetabolomics"]})
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None)
fig.show("png", width=1500, height=1000)

In [ ]:
import plotly.express as px
temp = results_df.assign(score = lambda x: x.module + "_" + x.features)
fig = px.scatter(temp, y="cindex", x="score", color="features", facet_col="endpoint", facet_col_wrap=5, template="plotly_dark",
               category_orders={"features": ["COX_Metabolomics", "DeepSurv_Metabolomics", 
                                             "COX_AgeSex", "DeepSurv_AgeSex",
                                             "COX_AgeSexMetabolomics", "DeepSurv_AgeSexMetabolomics"]})
fig.update_xaxes(matches=None)
#fig.update_yaxes(matches=None)
fig.show("png", width=1500, height=1000)

In [ ]:
import plotly.express as px
temp = results_df.assign(score = lambda x: x.module + "_" + x.features)
fig = px.violin(temp, y="cindex", x="features", color="features", box=True, points="all", facet_col="endpoint", facet_col_wrap=4,
               category_orders={"features": ["Metabolomics", "AgeSex", "AgeSexMetabolomics"]})
fig.update_xaxes(matches=None)
fig.show("png", width=1500, height=1000)

In [ ]:
preds.to_feather(f"{data_results_path}/predictions_model_210720.feather")

In [ ]:
from lifelines import RegressionFitter

In [ ]:
self._central_values = self._compute_central_values_of_raw_training_data(df, self.strata)
self.baseline_survival_ = self._compute_baseline_survival()
baseline_hazard_ = self._compute_baseline_hazards(predicted_partial_hazards_)
baseline_cumulative_hazard_ = self._compute_baseline_cumulative_hazard(baseline_hazard_)

In [ ]:
cph_semi = CoxPHFitter().fit(rossi, 'week', event_col='arrest')
cph_piecewise = CoxPHFitter(baseline_estimation_method="piecewise", breakpoints=[20, 35]).fit(rossi, 'week', event_col='arrest')

ax = cph_spline.baseline_cumulative_hazard_.plot()